# G05_eau — cahier autonome

Ce fichier contient aussi le moteur et les routines communes. Aucun fichier distant à charger. Exécuter de haut en bas. Votre routine remplace seulement celle de votre groupe dans la comparaison finale. Sauvegarder puis déposer le notebook sur OneDrive.

# Projet 5 Quand un océan devient possible

**Question.** Pourquoi ne suffit-il pas de dire « il fait moins de 100 °C » ? **Ancrage :** document 3, § 16–19, diagramme d’état. Lecture et interpolation.

**Entrées.** state : T_K ; diag : P_Pa, ph2o_Pa ; params : water_mode, choisi « pure_demo » ou « mixed_screen ». La routine ne retire aucun stock de vapeur.

**Données de frontière.** P en bar : 1, 10, 60, 100 ; températures d’ébullition arrondies en °C : 100, 180, 275, 311. Le point à 60 bar reprend la lecture du cours ; les autres valeurs arrondies sont ajoutées pour l’exercice. Les interpolations sont approximatives. Pas d’extrapolation hors 1–100 bar. Point triple : 0,01 °C et 610 Pa ; point critique : 374 °C et 221 bar.

**Modèle minimal.** En mode pure_demo, interpoler la température d’ébullition selon P et classer liquide si 273,15 K < T < Teb ; vapeur au-dessus ; frontière si égalité ; glace sous 273,15 K. C’est une approximation de l’eau pure limitée au domaine annoncé. Hors domaine, renvoyer « indetermine ».

**Mode du moteur.** En mélange, fixer ocean_possible = True seulement si T > 273,15 K et si ph2o_Pa dépasse la pression de saturation interpolée à T entre 100 et 311 °C. Aux autres températures, renvoyer None et water_domain = « indetermine ». Le test signale une possibilité de condensation, pas la quantité d’eau liquide. L’altération n’utilise True que dans ce domaine ou une valeur imposée explicitement par le scénario tempéré.

**Sorties.** diag : water_domain et ocean_possible. Aucun rates. En scénario tempéré, le professeur peut fournir un substitut « océan présent » avant l’assemblage du modèle de mélange.

**Validation.** Eau pure à 200 °C et 100 bar : liquide ; à 200 °C et 1 bar : vapeur. Tester une entrée hors domaine. Deux mélanges de même P et T mais de ph2o différent ne doivent pas être automatiquement classés de la même façon.

**Rendu.** Routine eau.py, diagramme et comparaison des deux modes. **Prolongement :** ajouter une table de saturation à basse température, puis un réservoir océanique conservant h2o total lors des transferts.

Lire aussi le contrat commun et le complément eau du dossier collectif.


In [ ]:
import types,sys
_module_classe=types.ModuleType("classe_commune")
_code_commun = 'VERSION_CLASSE = \'c41ab9f89aae\'\n"""Version commune de la classe : indépendante de tout fichier annexe."""\n\n\n"""Un moteur deterministe : lire, additionner les flux, avancer, verifier."""\nfrom copy import deepcopy\nfrom math import isfinite\n\nSTOCKS = ["co2_mol", "n2_mol", "o2_mol", "h2o_mol", "mantle_c_mol",\n          "rock_c_mol", "fe_mol", "oxide_fe_mol", "population", "fossil_J",\n          "emitted_c_mol"]\nVARIABLES = STOCKS + ["T_K", "ice"]\n\ndef verifier_etat(s):\n    for cle in VARIABLES + ["time_yr"]:\n        if not isfinite(s[cle]) or s[cle] < 0:\n            raise ValueError(f"Valeur invalide : {cle} = {s[cle]}")\n    if s["ice"] > 1 or s["T_K"] <= 0:\n        raise ValueError("Fraction de glace ou temperature invalide")\n\ndef verifier_diagnostic(x):\n    if isinstance(x, dict):\n        for valeur in x.values():\n            verifier_diagnostic(valeur)\n    elif isinstance(x, (int, float)) and not isfinite(x):\n        raise ValueError("Diagnostic non fini")\n    elif not isinstance(x, (str, bool, int, float, type(None))):\n        raise ValueError("Type de diagnostic invalide")\n\ndef evaluer(s, p, modules, dt):\n    travail = deepcopy(s)\n    travail["diag"] = {}\n    taux = {cle: 0.0 for cle in VARIABLES}\n    for module in modules:\n        try:\n            r = module(deepcopy(travail), deepcopy(p), dt)\n            if set(r) != {"rates", "values", "diag"} or r["values"]:\n                raise ValueError("Contrat de sortie incorrect")\n            for cle, valeur in r["rates"].items():\n                if cle not in taux or not isfinite(valeur):\n                    raise ValueError(f"Taux invalide : {cle}")\n                taux[cle] += valeur\n            for cle, valeur in r["diag"].items():\n                if cle in travail["diag"]:\n                    raise ValueError(f"Diagnostic duplique : {cle}")\n                verifier_diagnostic(valeur)\n                travail["diag"][cle] = valeur\n        except (ValueError, KeyError, TypeError) as erreur:\n            raise ValueError(f"{module.__module__}, t={s[\'time_yr\']}, dt={dt} : {erreur}") from erreur\n    return travail, taux\n\ndef planet_evolution(initial, params, modules, steps, dt_yr):\n    if not isfinite(dt_yr) or dt_yr <= 0:\n        raise ValueError("Le pas doit etre positif et fini")\n    if not isinstance(steps, int) or steps < 0:\n        raise ValueError("Le nombre de pas doit etre entier et >= 0")\n    s = deepcopy(initial)\n    verifier_etat(s)\n    histoire = []\n    for numero in range(steps + 1):\n        # On enregistre les diagnostics du MEME instant que les stocks.\n        travail, taux = evaluer(s, params, modules, dt_yr)\n        histoire.append(deepcopy(travail))\n        if numero == steps:\n            break\n        suivant = deepcopy(travail)\n        for cle in VARIABLES:\n            suivant[cle] = s[cle] + dt_yr * taux[cle]\n        suivant["time_yr"] += dt_yr\n        verifier_etat(suivant)\n        s = suivant\n    return histoire\n\n\n# Projet 1 : stocks en mol, pressions en Pa.\ndef routine_1(s, p, dt):\n    masses = {"co2_mol": .044, "n2_mol": .028,\n              "o2_mol": .032, "h2o_mol": .018}\n    total = sum(s[k] for k in masses)\n    masse = sum(s[k] * masses[k] for k in masses)\n    pression = masse * p["g_m_s2"] / p["area_m2"]\n    fractions = {}\n    for nom, cle in [("CO2", "co2_mol"), ("N2", "n2_mol"),\n                     ("O2", "o2_mol"), ("H2O", "h2o_mol")]:\n        fractions[nom] = s[cle] / total if total else 0.0\n    return {"rates": {}, "values": {}, "diag": {\n        "P_Pa": pression, "fractions": fractions,\n        "pco2_Pa": pression * fractions["CO2"],\n        "po2_Pa": pression * fractions["O2"],\n        "ph2o_Pa": pression * fractions["H2O"]}}\n\n\n# Projet 2 : un critere de retention, pas un taux de fuite.\ndef routine_2(s, p, dt):\n    vlib = (2 * p["G"] * p["mass_kg"] / p["radius_m"]) ** .5\n    retenus = {}\n    for gaz, masse_molaire in {"H2": .002, "N2": .028, "CO2": .044}.items():\n        masse = masse_molaire / p["NA"]\n        vitesse = (3 * p["kB"] * p["T_exo_K"] / masse) ** .5\n        retenus[gaz] = vlib > 6 * vitesse\n    return {"rates": {}, "values": {}, "diag": {\n        "vlib_m_s": vlib, "retention": retenus}}\n\n\n# Projet 3 : transfert conserve de carbone, mol/an.\ndef routine_3(s, p, dt):\n    flux = min(p["volcanic_mol_yr"], s["mantle_c_mol"] / dt)\n    return {"rates": {"co2_mol": flux, "mantle_c_mol": -flux},\n            "values": {}, "diag": {"volcanic_actual_mol_yr": flux}}\n\n\n# Projet 4 : equilibre radiatif sans serre.\ndef routine_4(s, p, dt):\n    albedo = .30 + .30 * s["ice"]\n    absorbe = p["S_W_m2"] * (1 - albedo) / 4\n    temperature = (absorbe / p["sigma"]) ** .25\n    return {"rates": {}, "values": {}, "diag": {\n        "albedo": albedo, "absorbed_W_m2": absorbe, "Teq_K": temperature}}\n\n\n# Projet 5 : interpolation, eau pure ou possibilite de condensation.\n# Table arrondie ajoutee au cours. Aucune masse d\'eau n\'est transferee.\nTABLE = [(0.01, 610), (10, 1230), (20, 2340), (30, 4240),\n         (50, 12350), (80, 47400), (100, 100000),\n         (180, 1000000), (275, 6000000), (311, 10000000)]\n\ndef interpoler(x, points):\n    for i in range(len(points) - 1):\n        x0, y0 = points[i]\n        x1, y1 = points[i + 1]\n        if x0 <= x <= x1:\n            return y0 + (y1 - y0) * (x - x0) / (x1 - x0)\n    return None\n\ndef routine_5(s, p, dt):\n    T = s["T_K"] - 273.15\n    pression = s["diag"]["P_Pa"]\n    saturation = interpoler(T, TABLE)\n    possible = None\n    domaine = "indetermine"\n    if p["water_mode"] == "pure_demo":\n        ebullition = interpoler(pression, [(v, t) for t, v in TABLE if v >= 1e5])\n        if ebullition is not None:\n            if T < 0:\n                domaine, possible = "glace", False\n            elif T == 0 or T == ebullition:\n                domaine = "frontiere"\n            elif T < ebullition:\n                domaine, possible = "liquide", True\n            else:\n                domaine, possible = "vapeur", False\n    elif saturation is not None:\n        possible = s["diag"]["ph2o_Pa"] >= saturation\n        domaine = "condensation_possible" if possible else "vapeur_non_saturee"\n    # Un ocean deja present est une condition prescrite du scenario.\n    # Son volume ne change pas : pas de simulation du cycle de l\'eau.\n    if p["ocean_prescribed"]:\n        possible = s["ice"] < 1 and 273.15 < s["T_K"] < 373.15\n        domaine = "ocean_prescrit" if possible else "ocean_prescrit_inactif"\n    return {"rates": {}, "values": {}, "diag": {\n        "water_domain": domaine, "ocean_possible": possible,\n        "saturation_Pa": saturation}}\n\n\n# Projet 6 : une loi fictive locale, transfert CO2 -> roches.\ndef routine_6(s, p, dt):\n    flux = 0.0\n    if s["diag"]["ocean_possible"] is True:\n        chaleur = max(0.0, 1 + (s["T_K"] - 288) / 50)\n        flux = (p["weather_mol_yr"] * p["land_fraction"]\n                * (1 - s["ice"]) * chaleur\n                * s["co2_mol"] / p["co2_ref_mol"])\n    flux = min(flux, s["co2_mol"] / dt)\n    return {"rates": {"co2_mol": -flux, "rock_c_mol": flux},\n            "values": {}, "diag": {"weather_actual_mol_yr": flux}}\n\n\n# Projet 7 : flux externe O2, puits fini de Fe ; stocks en mol.\ndef routine_7(s, p, dt):\n    production = p["oxygen_mol_yr"]\n    utilise = min(s["o2_mol"] + production * dt, s["fe_mol"] / 4)\n    return {"rates": {"o2_mol": production - utilise / dt,\n                       "fe_mol": -4 * utilise / dt,\n                       "oxide_fe_mol": 4 * utilise / dt},\n            "values": {}, "diag": {"oxygen_used_mol_yr": utilise / dt}}\n\n\n# Projet 8 : colonne d\'ozone prescrite, coefficient a une longueur d\'onde.\nfrom math import exp\n\ndef routine_8(s, p, dt):\n    photon = p["h"] * p["c"] / (p["lambda_nm"] * 1e-9)\n    seuil = 498000 / p["NA"]\n    transmission = exp(-p["absorption_m2_mol"] * p["ozone_column_mol_m2"])\n    return {"rates": {}, "values": {}, "diag": {\n        "photon_J": photon, "dissociation_possible": photon >= seuil,\n        "uv_fraction": transmission}}\n\n\n# Projet 9 : relaxation vers une couverture de glace fictive.\ndef routine_9(s, p, dt):\n    if dt > p["tau_ice_yr"]:\n        raise ValueError("dt doit etre <= tau_ice_yr")\n    if s["T_K"] <= 263:\n        cible = 1.0\n    elif s["T_K"] >= 283:\n        cible = 0.0\n    else:\n        cible = (283 - s["T_K"]) / 20\n    taux = (cible - s["ice"]) / p["tau_ice_yr"]\n    return {"rates": {"ice": taux}, "values": {}, "diag": {}}\n\n\n# Projet 10 : loi de serre fictive, Euler explicite.\ndef routine_10(s, p, dt):\n    if dt > p["tau_T_yr"]:\n        raise ValueError("dt doit etre <= tau_T_yr")\n    co2 = s["diag"]["pco2_Pa"]\n    serre = 66 * co2 / (co2 + p["pref_Pa"])\n    cible = s["diag"]["Teq_K"] + serre\n    taux = (cible - s["T_K"]) / p["tau_T_yr"]\n    return {"rates": {"T_K": taux}, "values": {},\n            "diag": {"Ttarget_K": cible, "greenhouse_K": serre}}\n\n\n# Projet 11 : energies annuelles en J/an, reserves en J.\ndef routine_11(s, p, dt):\n    demande = s["population"] * p["need_J_person_yr"]\n    renouvelable = min(demande, p["renewable_J_yr"])\n    fossile = min(max(0.0, demande - renouvelable),\n                  p["fossil_capacity_J_yr"], s["fossil_J"] / dt)\n    fourni = renouvelable + fossile\n    deficit = (demande - fourni) / demande if demande else 0.0\n    emissions = fossile * p["carbon_mol_J"]\n    return {"rates": {"fossil_J": -fossile, "co2_mol": emissions,\n                       "emitted_c_mol": emissions},\n            "values": {}, "diag": {"demand_J_yr": demande,\n            "supplied_J_yr": fourni, "deficit_fraction": deficit,\n            "power_W": fourni / 31557600}}\n\n\n# Projet 12 : population moyenne continue, taux en 1/an.\ndef routine_12(s, p, dt):\n    naissances = p["birth_rate_yr"] * s["population"]\n    deces = (p["death_rate_yr"] + p["deficit_death_rate_yr"]\n             * s["diag"]["deficit_fraction"]) * s["population"]\n    return {"rates": {"population": naissances - deces}, "values": {},\n            "diag": {"births_yr": naissances, "deaths_yr": deces}}\n\n\nROUTINES = {1:routine_1, 2:routine_2, 3:routine_3, 4:routine_4, 5:routine_5, 6:routine_6, 7:routine_7, 8:routine_8, 9:routine_9, 10:routine_10, 11:routine_11, 12:routine_12}\nORDRE = [1, 2, 4, 5, 8, 11, 3, 6, 7, 9, 10, 12]\nMODULES = [ROUTINES[i] for i in ORDRE]\n\n\ndef base():\n    s = {"time_yr": 0., "T_K": 290., "ice": 0.,\n         "co2_mol": 7.49e16, "n2_mol": 1.39e20, "o2_mol": 3.74e19,\n         "h2o_mol": 0., "mantle_c_mol": 1e20, "rock_c_mol": 0.,\n         "fe_mol": 0., "oxide_fe_mol": 0., "population": 0.,\n         "fossil_J": 0., "emitted_c_mol": 0., "diag": {}}\n    p = {"g_m_s2": 9.81, "area_m2": 5.10e14, "G": 6.67e-11,\n         "kB": 1.38e-23, "NA": 6.02e23, "h": 6.63e-34, "c": 3e8,\n         "mass_kg": 5.97e24, "radius_m": 6.371e6, "T_exo_K": 1000.,\n         "S_W_m2": 1361., "sigma": 5.67e-8, "pref_Pa": 42.,\n         "tau_T_yr": 10., "tau_ice_yr": 20.,\n         "water_mode": "mixed_screen", "ocean_prescribed": True,\n         "volcanic_mol_yr": 0., "weather_mol_yr": 0.,\n         "land_fraction": 1., "co2_ref_mol": 7.49e16,\n         "oxygen_mol_yr": 0., "lambda_nm": 255.,\n         "ozone_column_mol_m2": .10, "absorption_m2_mol": 100.,\n         "need_J_person_yr": 1e9, "renewable_J_yr": 4e11,\n         "fossil_capacity_J_yr": 3e11, "carbon_mol_J": 2e-6,\n         "birth_rate_yr": 0., "death_rate_yr": 0., "deficit_death_rate_yr": 0.}\n    return s, p\n\ndef scenario(nom):\n    s, p = base()\n    duree, dt = 500., 1.\n    if nom == "chaude":\n        pass\n    elif nom == "gelee":\n        s.update(T_K=250., ice=1.)\n    elif nom == "deglaciation":\n        s.update(T_K=250., ice=1.)\n        p.update(volcanic_mol_yr=1e13, weather_mol_yr=1e13,\n                 tau_T_yr=10000., tau_ice_yr=20000.)\n        duree, dt = 1000000., 1000.\n    elif nom == "societe":\n        s.update(population=1000., fossil_J=1e13)\n        p.update(birth_rate_yr=.02, death_rate_yr=.01,\n                 deficit_death_rate_yr=.02)\n        duree, dt = 100., 1.\n    elif nom == "oxygenation":\n        s.update(o2_mol=0., fe_mol=40.)\n        p.update(oxygen_mol_yr=1.)\n        duree, dt = 20., 1.\n    else:\n        raise ValueError("Scenario inconnu : " + nom)\n    return s, p, int(duree / dt), dt\n\n# Projet 13 : le hasard est a l\'exterieur du moteur.\nfrom copy import deepcopy\nfrom random import Random\n\n\ndef quantile(valeurs, fraction):\n    ordre = sorted(valeurs)\n    position = fraction * (len(ordre) - 1)\n    i = int(position)\n    j = min(i + 1, len(ordre) - 1)\n    return ordre[i] + (ordre[j] - ordre[i]) * (position - i)\n\ndef ensemble(initial, params, modules, steps, dt, nombre=50, seed=2026,\n             bornes=None):\n    if nombre < 1:\n        raise ValueError("Au moins une simulation")\n    if bornes is None:\n        bornes = {"tau_T_yr": (5., 15.), "pref_Pa": (30., 60.)}\n    rng = Random(seed)\n    resultats = {"seed": seed, "samples": [], "histories": [],\n                 "summary": {}, "failures": []}\n    for _ in range(nombre):\n        p = deepcopy(params)\n        for cle, (bas, haut) in bornes.items():\n            if bas > haut:\n                raise ValueError("Bornes inversees")\n            p[cle] = rng.uniform(bas, haut)\n        resultats["samples"].append(p)\n        try:\n            h = planet_evolution(initial, p, modules, steps, dt)\n            resultats["histories"].append(h)\n        except ValueError as erreur:\n            resultats["histories"].append(None)\n            resultats["failures"].append({"params": p, "error": str(erreur)})\n    bonnes = [h for h in resultats["histories"] if h is not None]\n    if bonnes:\n        T = [h[-1]["T_K"] for h in bonnes]\n        resultats["summary"] = {\n            "successes": len(T), "q05_T_K": quantile(T, .05),\n            "median_T_K": quantile(T, .5), "q95_T_K": quantile(T, .95),\n            "frozen_fraction": sum(h[-1]["ice"] > .9 for h in bonnes) / len(T)}\n    return resultats\n\n\nfrom math import pi\nMONDE = {\'nom\': \'Asteria\', \'luminosite_W\': 3.828e+26, \'distance_m\': 149600000000.0, \'masse_kg\': 5.97e+24, \'rayon_m\': 6371000.0, \'duree_yr\': 5000000000.0, \'stellar_validity_yr\': 10000000000.0, \'stellar_initial_ratio\': 0.75, \'stellar_final_ratio\': 1.0, \'water_kg\': 1.4e+21, \'co2_initial_mol\': 3e+19, \'n2_mol\': 1.39e+20}\n\n\nORIGINES = {\'1\': {\'statut\': \'référence professeur\', \'fichier\': \'moteur/modules/gaz.py\'}, \'2\': {\'statut\': \'référence professeur\', \'fichier\': \'moteur/modules/retention.py\'}, \'3\': {\'statut\': \'référence professeur\', \'fichier\': \'moteur/modules/degazage.py\'}, \'4\': {\'statut\': \'référence professeur\', \'fichier\': \'moteur/modules/radiatif.py\'}, \'5\': {\'statut\': \'référence professeur\', \'fichier\': \'moteur/modules/eau.py\'}, \'6\': {\'statut\': \'référence professeur\', \'fichier\': \'moteur/modules/alteration.py\'}, \'7\': {\'statut\': \'référence professeur\', \'fichier\': \'moteur/modules/oxygenation.py\'}, \'8\': {\'statut\': \'référence professeur\', \'fichier\': \'moteur/modules/uv.py\'}, \'9\': {\'statut\': \'référence professeur\', \'fichier\': \'moteur/modules/glace.py\'}, \'10\': {\'statut\': \'référence professeur\', \'fichier\': \'moteur/modules/climat.py\'}, \'11\': {\'statut\': \'référence professeur\', \'fichier\': \'moteur/modules/energie.py\'}, \'12\': {\'statut\': \'référence professeur\', \'fichier\': \'moteur/modules/demographie.py\'}, \'13\': {\'statut\': \'référence professeur\', \'fichier\': \'moteur/ensemble.py\'}}\n\n\n# Ce support est inclus dans le module autonome ; il ne change pas les routines.\nfrom html import escape\nimport json\n\ndef scenario_classe(nom=\'societe\'):\n    s,p,n,dt=scenario(nom)\n    p.update(S_W_m2=MONDE[\'luminosite_W\']/(4*pi*MONDE[\'distance_m\']**2),\n             g_m_s2=6.67e-11*MONDE[\'masse_kg\']/MONDE[\'rayon_m\']**2,\n             area_m2=4*pi*MONDE[\'rayon_m\']**2,\n             mass_kg=MONDE[\'masse_kg\'],radius_m=MONDE[\'rayon_m\'])\n    return s,p,n,dt\n\ndef comparer_groupe(groupe,routine,nom=\'societe\'):\n    """La routine locale remplace un groupe ; les autres restent communes."""\n    s,p,n,dt=scenario_classe(nom)\n    if groupe not in ROUTINES:raise ValueError(\'Groupe 1 à 12 attendu\')\n    # Banc de raccordement : vérifier la sortie aux mêmes diagnostics que les voisins.\n    avec_diag,_=evaluer(s,p,MODULES,dt)\n    diagnostic,params=deepcopy(avec_diag),deepcopy(p)\n    r=routine(diagnostic,params,dt)\n    if diagnostic!=avec_diag or params!=p:raise ValueError(\'La routine modifie les entrées\')\n    if set(r)!={\'rates\',\'values\',\'diag\'} or r[\'values\']:raise ValueError(\'Contrat incorrect\')\n    autres=[routine if i==groupe else ROUTINES[i] for i in ORDRE]\n    commun=planet_evolution(s,p,MODULES,n,dt)\n    local=planet_evolution(s,p,autres,n,dt)\n    return {\'version_classe\':VERSION_CLASSE,\'groupe\':groupe,\'scenario\':nom,\n            \'monde\':deepcopy(MONDE),\'origines\':deepcopy(ORIGINES),\n            \'commun\':commun,\'avec_ma_routine\':local}\n\ndef rapport_comparaison(resultats):\n    a=resultats[\'commun\'];b=resultats[\'avec_ma_routine\'];D=a[-1][\'time_yr\'] or 1\n    figures=[]\n    for k,titre,unite in [(\'T_K\',\'Température\',\'K\'),(\'co2_mol\',\'CO2 atmosphérique\',\'mol\'),\n                          (\'ice\',\'Fraction de glace\',\'fraction\'),(\'population\',\'Population moyenne\',\'personnes\')]:\n        vals=[s[k] for h in [a,b] for s in h];lo,hi=min(vals),max(vals)\n        marge=max((hi-lo)*.05,abs(hi)*1e-6,1);lo-=marge;hi+=marge\n        if k==\'ice\':lo,hi=0.,1.\n        lines=[]\n        for h,color,dash in [(a,\'#81949c\',\' stroke-dasharray="6 4"\'),(b,\'#126478\',\'\')]:\n            points=\' \'.join(f\'{80+400*s["time_yr"]/D:.2f},{190-125*(s[k]-lo)/(hi-lo):.2f}\' for s in h)\n            lines.append(f\'<polyline points="{points}" fill="none" stroke="{color}" stroke-width="2"{dash}/>\')\n        figures.append(f\'<svg viewBox="0 0 520 245" role="img" aria-label="{titre}"><text x="80" y="25">{titre} ({unite})</text><path d="M80 60V190H480" stroke="#ccc" fill="none"/><text x="3" y="65">{hi:.4g}</text><text x="3" y="190">{lo:.4g}</text>{"".join(lines)}<text x="80" y="213">0</text><text x="435" y="213">{D:g} ans</text><text x="195" y="240">Temps depuis le début (an)</text></svg>\')\n    origine=\' ; \'.join(\'G\'+str(i)+\' : \'+v[\'statut\'] for i,v in sorted(((int(k),v) for k,v in resultats[\'origines\'].items())))\n    return f\'\'\'<!doctype html><html lang="fr"><meta charset="utf-8"><title>Notre contribution dans le monde commun</title><style>body{{font:16px/1.5 system-ui;max-width:1000px;margin:30px auto;padding:20px;color:#193442}}svg{{width:100%;font:12px system-ui}}.figures{{display:grid;grid-template-columns:1fr 1fr}}@media print{{button{{display:none}}}}@media(max-width:600px){{.figures{{grid-template-columns:1fr}}}}</style><h1>Notre contribution dans le monde commun</h1><p>Groupe {resultats[\'groupe\']} · scénario {escape(resultats[\'scenario\'])} · monde {escape(resultats[\'monde\'][\'nom\'])} · version commune {escape(resultats[\'version_classe\'])}</p><button onclick="window.print()">Imprimer / PDF</button><p>Bleu : avec notre routine. Gris pointillé : version commune. Toutes les autres routines sont identiques dans les deux calculs.</p><div class="figures">{\'\'.join(figures)}</div><h2>Ce que nous expliquons</h2><p>Notre changement : … ; effet attendu : … ; effet observé : … ; mécanisme : … ; limite : …</p><h2>Versions des contributions</h2><p>{escape(origine)}</p><p>Le cycle hydrologique géologique appartient à l’autre moteur ; ce raccordement annuel n’en simule pas les stocks. Les courbes ne démontrent pas l’habitabilité.</p></html>\'\'\'\n\n'
exec(_code_commun,_module_classe.__dict__)
sys.modules["classe_commune"]=_module_classe
print("Version commune :",_module_classe.VERSION_CLASSE)


## Travailler dans le navigateur

Ouvrir ce fichier .ipynb dans https://notebook.basthon.fr ou Colab. Aucun fichier Python annexe ni installation locale ne sont requis. Exécuter les cellules de haut en bas avec Maj + Entrée. Compléter uniquement les zones **À VOUS**, puis relancer les cellules qui suivent. Télécharger le notebook à la fin et le déposer dans votre dossier OneDrive G05. Le notebook ne se sauvegarde pas automatiquement dans OneDrive.

Les fonctions de référence du banc d’essai sont fournies ; elles ne sont pas la routine de votre groupe. Les formules sont dans la fiche ci-dessus et le complément commun.

## À VOUS — raisonnement avant le code

Membres et rôles : …

Entrées, sorties et unités : …

Formule symbolique et hypothèses : …

Calcul manuel : données … ; résultat attendu …

Cas limite prévu : …

In [ ]:
from copy import deepcopy
from math import exp, pi, isfinite, isclose
from random import Random
import matplotlib.pyplot as plt
def base():
    s = {"time_yr": 0., "T_K": 290., "ice": 0.,
         "co2_mol": 7.49e16, "n2_mol": 1.39e20, "o2_mol": 3.74e19,
         "h2o_mol": 0., "mantle_c_mol": 1e20, "rock_c_mol": 0.,
         "fe_mol": 0., "oxide_fe_mol": 0., "population": 0.,
         "fossil_J": 0., "emitted_c_mol": 0., "diag": {}}
    p = {"g_m_s2": 9.81, "area_m2": 5.10e14, "G": 6.67e-11,
         "kB": 1.38e-23, "NA": 6.02e23, "h": 6.63e-34, "c": 3e8,
         "mass_kg": 5.97e24, "radius_m": 6.371e6, "T_exo_K": 1000.,
         "S_W_m2": 1361., "sigma": 5.67e-8, "pref_Pa": 42.,
         "tau_T_yr": 10., "tau_ice_yr": 20.,
         "water_mode": "mixed_screen", "ocean_prescribed": True,
         "volcanic_mol_yr": 0., "weather_mol_yr": 0.,
         "land_fraction": 1., "co2_ref_mol": 7.49e16,
         "oxygen_mol_yr": 0., "lambda_nm": 255.,
         "ozone_column_mol_m2": .10, "absorption_m2_mol": 100.,
         "need_J_person_yr": 1e9, "renewable_J_yr": 4e11,
         "fossil_capacity_J_yr": 3e11, "carbon_mol_J": 2e-6,
         "birth_rate_yr": 0., "death_rate_yr": 0., "deficit_death_rate_yr": 0.}
    return s, p
"""Un moteur deterministe : lire, additionner les flux, avancer, verifier."""
from copy import deepcopy
from math import isfinite

STOCKS = ["co2_mol", "n2_mol", "o2_mol", "h2o_mol", "mantle_c_mol",
          "rock_c_mol", "fe_mol", "oxide_fe_mol", "population", "fossil_J",
          "emitted_c_mol"]
VARIABLES = STOCKS + ["T_K", "ice"]

def verifier_etat(s):
    for cle in VARIABLES + ["time_yr"]:
        if not isfinite(s[cle]) or s[cle] < 0:
            raise ValueError(f"Valeur invalide : {cle} = {s[cle]}")
    if s["ice"] > 1 or s["T_K"] <= 0:
        raise ValueError("Fraction de glace ou temperature invalide")

def verifier_diagnostic(x):
    if isinstance(x, dict):
        for valeur in x.values():
            verifier_diagnostic(valeur)
    elif isinstance(x, (int, float)) and not isfinite(x):
        raise ValueError("Diagnostic non fini")
    elif not isinstance(x, (str, bool, int, float, type(None))):
        raise ValueError("Type de diagnostic invalide")

def evaluer(s, p, modules, dt):
    travail = deepcopy(s)
    travail["diag"] = {}
    taux = {cle: 0.0 for cle in VARIABLES}
    for module in modules:
        try:
            r = module(deepcopy(travail), deepcopy(p), dt)
            if set(r) != {"rates", "values", "diag"} or r["values"]:
                raise ValueError("Contrat de sortie incorrect")
            for cle, valeur in r["rates"].items():
                if cle not in taux or not isfinite(valeur):
                    raise ValueError(f"Taux invalide : {cle}")
                taux[cle] += valeur
            for cle, valeur in r["diag"].items():
                if cle in travail["diag"]:
                    raise ValueError(f"Diagnostic duplique : {cle}")
                verifier_diagnostic(valeur)
                travail["diag"][cle] = valeur
        except (ValueError, KeyError, TypeError) as erreur:
            raise ValueError(f"{module.__module__}, t={s['time_yr']}, dt={dt} : {erreur}") from erreur
    return travail, taux

def planet_evolution(initial, params, modules, steps, dt_yr):
    if not isfinite(dt_yr) or dt_yr <= 0:
        raise ValueError("Le pas doit etre positif et fini")
    if not isinstance(steps, int) or steps < 0:
        raise ValueError("Le nombre de pas doit etre entier et >= 0")
    s = deepcopy(initial)
    verifier_etat(s)
    histoire = []
    for numero in range(steps + 1):
        # On enregistre les diagnostics du MEME instant que les stocks.
        travail, taux = evaluer(s, params, modules, dt_yr)
        histoire.append(deepcopy(travail))
        if numero == steps:
            break
        suivant = deepcopy(travail)
        for cle in VARIABLES:
            suivant[cle] = s[cle] + dt_yr * taux[cle]
        suivant["time_yr"] += dt_yr
        verifier_etat(suivant)
        s = suivant
    return histoire

def simuler_isole(initial, params, routine, steps, dt):
    """Banc d'essai : diagnostics voisins prescrits et conservés.
    Ce banc ne remplace pas le moteur collectif, qui recalcule les voisins.
    """
    s=deepcopy(initial);h=[deepcopy(s)]
    for _ in range(steps):
        r=routine(deepcopy(s),deepcopy(params),dt)
        assert set(r)=={'rates','values','diag'} and not r['values']
        for cle,taux in r['rates'].items():
            s[cle]+=dt*taux
            assert isfinite(s[cle]) and s[cle]>=0,cle
        s['time_yr']+=dt
        if 'ice' in s:assert s['ice']<=1
        h.append(deepcopy(s))
    return h


In [ ]:
# Projet 5 : interpolation, eau pure ou possibilite de condensation.
# Table arrondie ajoutee au cours. Aucune masse d'eau n'est transferee.
TABLE = [(0.01, 610), (10, 1230), (20, 2340), (30, 4240),
         (50, 12350), (80, 47400), (100, 100000),
         (180, 1000000), (275, 6000000), (311, 10000000)]

def interpoler(x, points):
    for i in range(len(points) - 1):
        x0, y0 = points[i]
        x1, y1 = points[i + 1]
        if x0 <= x <= x1:
            return y0 + (y1 - y0) * (x - x0) / (x1 - x0)
    return None



## À VOUS — écrire la routine

Lire les entrées de la fiche, calculer la relation et traiter les cas limites. Retourner rates, values vide et diag avec les clés exactes du contrat.

Remplacer le `raise NotImplementedError` par votre code. Les tests indiquent les valeurs attendues ; ils ne donnent pas votre implémentation.

In [ ]:
def step(s, p, dt):
    # À VOUS : écrire ici le modèle donné dans la fiche.
    raise NotImplementedError("routine du groupe 5")


## Vérifier votre fonction

Avant de coder, la cellule affiche « À compléter ». Une fois la routine écrite, une assertion qui échoue signifie que le test doit être examiné. Noter **attendu / obtenu / conclusion**, même quand les assertions passent.

In [ ]:
try:
    s,p=base();p.update(water_mode='pure_demo',ocean_prescribed=False)
    s['T_K']=473.15;s['diag']={'P_Pa':1e7,'ph2o_Pa':1e7}
    assert step(s,p,1)['diag']['ocean_possible'] is True
    s['diag']['P_Pa']=1e5;assert step(s,p,1)['diag']['ocean_possible'] is False
    p['water_mode']='mixed_screen';s['T_K']=293.15;s['diag'].update(P_Pa=1e5,ph2o_Pa=1000)
    assert step(s,p,1)['diag']['ocean_possible'] is False
    s['diag']['ph2o_Pa']=3000;assert step(s,p,1)['diag']['ocean_possible'] is True
    s['T_K']=1000;assert step(s,p,1)['diag']['ocean_possible'] is None
    print("Vérifications numériques réussies. Ajouter votre propre essai ci-dessous.")
    s,p=base(); s0,p0=deepcopy(s),deepcopy(p)
    s['diag']={'P_Pa':1e5,'ph2o_Pa':1000}
    s0,p0=deepcopy(s),deepcopy(p);step(s,p,1)
    assert s==s0 and p==p0,"La routine a modifié les entrées"
except NotImplementedError as erreur:
    print("À compléter :", erreur)


## À VOUS — ajouter une vérification justifiée

Quel paramètre changez-vous ? Quel résultat prévoyez-vous et pourquoi ?

In [ ]:
# À VOUS : ajouter un test avec un résultat attendu et une justification.


## Première figure fournie comme point de départ

La cellule propose une expérience et des axes. Exécutez-la après avoir codé votre routine. Ensuite, modifiez un paramètre, comparez deux expériences et justifiez votre lecture. Ce graphique isolé n’est pas encore la planète collective.

In [ ]:
try:
    plt.figure(figsize=(7,4))
    s,p=base();p.update(water_mode='mixed_screen',ocean_prescribed=False)
    x=list(range(1,101));y=[]
    for T in x:
     q=deepcopy(s);q['T_K']=T+273.15;q['diag']={'P_Pa':1e5,'ph2o_Pa':1000}
     y.append(step(q,p,1)['diag']['saturation_Pa'])
    plt.plot(x,y);plt.xlabel('Température (°C)');plt.ylabel('Pression de saturation (Pa)')
    plt.title("Groupe 5 — eau")
    plt.grid(alpha=.25)
    plt.show()
except NotImplementedError as erreur:
    print("À compléter :", erreur)


## À VOUS — interprétation et raccordement

Prédiction avant la figure : …

Ce que montre la courbe : …

Explication par la formule : …

Limite de notre modèle : …

Voisin fournisseur et donnée de substitution à retirer : …

Sortie utilisée par un autre groupe : …

Contribution au graphe collectif et signe de la rétroaction : …

Livrer ce notebook complété, avec les résultats visibles et le nom des membres.

## Extension pluie et océans — deuxième étape guidée

Le projet 5 étudie d'abord le diagnostic de phase de sa fiche ; le complément suivant fournit toutes les lois de l'extension géologique. Elles sont pédagogiques. Il n'est pas demandé aux élèves de les deviner.

Entrées : T (K), fraction de glace i entre 0 et 1, vapeur V (kg), eau totale Wtot (kg), surface A (m²), pas dt (an). Paramètres : E0 = 5 × 10¹⁷ kg/an et tau = 0,02 an dans l'exemple. Ces valeurs se modifient avec le scénario.

Évaporation : E = E0 × (1 − i) × f(T), avec f(T) = exp[(T − 288)/30], borné entre 0,05 et 3. L'exponentielle est donnée et disponible par math.exp ; elle ne constitue pas un prérequis de cours. À 288 K et sans glace, E = E0 ; avec i = 1, E = 0.

Vapeur : dV/dt = E − V/tau. Sur un pas à E constant : Vnouveau = E tau + (Vancien − E tau) exp(−dt/tau). Le code limite la cible à Wtot. Précipitation moyenne : P = E − (Vnouveau − Vancien)/dt. Le débit instantané est V/tau. Distinguer ces deux valeurs dans les résultats.

Eau de surface = Wtot − V. Océan liquide = (1 − i)(Wtot − V). Eau glacée = i(Wtot − V). Bilan à vérifier : vapeur + océan + glace = Wtot. La couverture i est choisie comme proxy de la fraction massique gelée ; le modèle ne calcule pas les calottes ni la chaleur latente.

Pluie moyenne = P(1 − i)/A en mm/an ; neige = Pi/A en mm équivalent eau/an. 1 kg/m² d'eau correspond à 1 mm. À 288 K et i = 0, l'équilibre vapeur vaut E0 tau = 10¹⁶ kg. Pour A = 5,10 × 10¹⁴ m², la pluie vaut environ 980 mm/an. Ce résultat est une moyenne planétaire, pas la météo locale.

Altération du moteur géologique : k = (F0/Cref) × land × max[0, 1 + (T − 288)/50] × (P/E0) × (1 − i). Flux retiré instantané = kC. Avec F volcanique constant : Cnouveau = C exp(−kdt) + (F/k)[1 − exp(−kdt)] ; si k = 0, Cnouveau = C + Fdt. Le retrait est C + Fdt − Cnouveau ; la même quantité est ajoutée aux roches. Fdt est retiré du manteau. Les coefficients sont figés sur le pas, d'où la nécessité du test de convergence.

Figure : vapeur, océan et glace dans le temps ; pluie dans le temps ; comparer avec et sans action de la pluie sur l'altération. Tests : bilan eau, bilan du flux, état glacé et équilibre à 288 K. La boucle à expliquer : T augmente → évaporation et pluie augmentent → altération augmente → CO2 diminue → effet de serre diminue → T diminue.

Cette extension utilise des solutions analytiques et un équilibre climatique rapide. Elle ne se branche pas telle quelle comme taux dans le moteur annuel : ses stocks, son traitement du temps et ses unités sont décrits dans histoire_planete.py. L'assemblage des élèves commence avec le contrat annuel ; le professeur présente ensuite la réduction géologique.


In [ ]:
def hydrologie(T, ice, vapor, dt, p):
    # À VOUS : appliquer les relations fournies ci-dessus.
    # Retour : vapor_kg, ocean_kg, ice_kg, evap_kg_yr, precip_kg_yr,
    # rain_mm_yr et snow_mm_yr. Les débits sont moyennés sur le pas.
    raise NotImplementedError("cycle de l’eau")


In [ ]:
try:
    p={'water_kg':1.4e21,'rain_ref_kg_yr':5e17,'vapor_tau_yr':.02,'area_m2':5.10e14}
    r=hydrologie(288,0,1e16,1,p)
    assert isclose(r['vapor_kg'],1e16)
    assert isclose(r['rain_mm_yr'],5e17/5.10e14)
    assert isclose(r['vapor_kg']+r['ocean_kg']+r['ice_kg'],p['water_kg'])
    r=hydrologie(260,1,1e16,1,p)
    assert r['ocean_kg']==0 and r['rain_mm_yr']==0
    print('Bilans et cas limites du cycle de l’eau réussis.')
except NotImplementedError as erreur:
    print("À compléter :", erreur)


## Tester notre routine dans le projet commun

Le lien de classe Basthon charge aussi `classe_commune.py` depuis GitHub. Les autres groupes proviennent de la version commune affichée. Votre fonction locale remplace uniquement votre groupe, sans changer le dépôt. Si vous avez ouvert seulement le fichier .ipynb, charger aussi classe_commune.py comme module avec le bouton d’ouverture.

Une sauvegarde Basthon télécharge votre copie ; elle ne publie pas votre travail sur GitHub. Déposer le notebook et les résultats sur OneDrive. Après validation, le professeur met à jour la version commune ; les autres groupes la récupèrent au prochain chargement. Les références professeur restent identifiées tant qu’une routine élève n’a pas été validée.


In [ ]:
import json
try:
    from classe_commune import comparer_groupe, rapport_comparaison, VERSION_CLASSE
    print("Version commune :",VERSION_CLASSE)
    # Choix : chaude, gelee, deglaciation, societe, oxygenation.
    # Choisir un scénario qui sollicite votre mécanisme.
    resultat_classe=comparer_groupe(5,step,nom="societe")
    page=rapport_comparaison(resultat_classe)
    from IPython.display import HTML, display
    from base64 import b64encode
    from html import escape
    display(HTML('<iframe title="Notre contribution" style="width:100%;height:900px;border:0" sandbox="allow-scripts allow-modals" srcdoc="'+escape(page,quote=True)+'"></iframe>'))
    for contenu,nom in [(page,"G05_rapport.html"),(json.dumps(resultat_classe,ensure_ascii=False),"G05_resultats.json")]:
        encoded=b64encode(contenu.encode()).decode()
        display(HTML('<p><a download="'+nom+'" href="data:application/octet-stream;base64,'+encoded+'">Télécharger '+nom+'</a></p>'))
except ImportError:
    print("Ouvrir le lien Basthon de classe, ou charger classe_commune.py comme module.")
except NotImplementedError as erreur:
    print("Compléter d’abord votre routine :",erreur)


## Notre bilan collectif

Quelle version commune avons-nous utilisée ? …

Quelles routines viennent de nos camarades et lesquelles restent des références ? …

Quel effet produit notre routine dans ce scénario ? …

Déposer le notebook, le rapport et les résultats dans le dossier du groupe.